# 02 — Cleaning and feature engineering

Two jobs here:

1. **Repair the extracts** — and count every repair, so the work is auditable rather than
   invisible.
2. **Engineer a modelling table with a time cutoff** — features from months 1–21, label from
   months 22–24. Nothing that happens after the cutoff can leak into a feature.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.color": "#EAE6DF",
                     "axes.facecolor": "white", "figure.facecolor": "white",
                     "axes.edgecolor": "#C9C4BA", "axes.axisbelow": True,
                     "legend.frameon": False})
SLATE, CLAY, AMBER = "#335C67", "#9E2A2B", "#E09F3E"


In [ ]:
from preprocessing import (load_raw, clean_all, build_panel, build_features,
                           FEATURE_COLUMNS_NUMERIC, FEATURE_COLUMNS_CATEGORICAL)

raw = load_raw()
{k: v.shape for k, v in raw.items()}

## The defects, before

In [ ]:
cust_raw = raw["customers"]
print(f"Rows:                {len(cust_raw):,}")
print(f"Duplicate IDs:       {cust_raw.customer_id.duplicated().sum():,}")
print(f"Gender variants:     {cust_raw.gender.nunique()} -> {sorted(cust_raw.gender.dropna().unique())}")
print(f"Ages outside 16-100: {((cust_raw.age < 16) | (cust_raw.age > 100)).sum():,}")
print(f"Missing income:      {cust_raw.income_level.isna().sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].hist(cust_raw.age.dropna(), bins=50, color=CLAY)
axes[0].set_title("Age, before cleaning")
axes[1].hist(raw["usage"].usage_hours.dropna(), bins=80, color=CLAY)
axes[1].set_yscale("log")
axes[1].set_title("Usage hours, before cleaning (log count)")
plt.tight_layout()

Two obvious problems: ages clustered at 0, 1, 199 and 250 (data-entry errors), and a long
tail of usage readings that are physically impossible — more hours than exist in a month.

Outliers are **winsorised at the 99.9th percentile rather than deleted**. The customer-month
still happened; only the magnitude is untrustworthy. Dropping the row would quietly remove a
real customer from the denominator of every churn rate downstream.

In [ ]:
clean, report = clean_all(raw)

pd.Series(report, name="count").to_frame().style.format("{:,.0f}")

In [ ]:
cust = clean["customers"]
print("Gender after mapping:  ", sorted(cust.gender.unique()))
print("Payment methods after: ", sorted(clean["payments"].payment_method.unique()))
print(f"Rows after dedupe:      {len(cust):,}")
print(f"Ages now in range:      {cust.age.min()}-{cust.age.max()}")

## The customer-month panel

Engagement, billing and ticket detail joined to one row per customer per month. This is the
grain everything else is aggregated from.

In [ ]:
panel = build_panel(clean)
print(f"{len(panel):,} customer-months, {panel.customer_id.nunique():,} customers")
panel.head(4)

## The cutoff that prevents leakage

This is the single most important design decision in the project.

```
months 1 .............. 21 | 22  23  24
   features built here     |  label observed here
                        cutoff
```

A customer enters the training table only if they were **still active at the end of month
21**. Their features summarise months 1–21. Their label is whether they cancelled in months
22–24. Because the windows do not overlap, no feature can encode the outcome.

The exact same function, pointed at the last month with `horizon=None`, produces the
**scoring table**: every customer active today, with no label. That is what the model runs
against in production — same code path, so training and serving cannot drift apart.

In [ ]:
train_tbl = build_features(clean, panel, cutoff_index=20, horizon=3)
score_tbl = build_features(clean, panel, cutoff_index=int(clean["dim_date"].month_index.max()), horizon=None)

print(f"Training table: {len(train_tbl):,} customers active at the cutoff")
print(f"  churn in the next 3 months: {train_tbl.churned.mean():.2%} "
      f"({int(train_tbl.churned.sum()):,} customers)")
print(f"Scoring table:  {len(score_tbl):,} customers active today")

A 5.5% positive rate is a realistic, awkward class balance — and it is the reason accuracy
is a useless metric here. A model that predicts "nobody churns" scores 94.5%.

In [ ]:
features = FEATURE_COLUMNS_NUMERIC + FEATURE_COLUMNS_CATEGORICAL
print(f"{len(features)} features: {len(FEATURE_COLUMNS_NUMERIC)} numeric, "
      f"{len(FEATURE_COLUMNS_CATEGORICAL)} categorical\n")

groups = {
    "Tenure & contract": ["tenure_months", "months_active", "is_month_to_month", "new_customer_flag"],
    "Engagement level":  ["usage_hours_mean", "usage_hours_l3", "logins_l3", "feature_usage_l3"],
    "Engagement trend":  ["usage_trend_3m", "login_trend_3m", "usage_vs_lifetime", "usage_volatility"],
    "Support pressure":  ["tickets_l3", "tickets_l6", "support_frequency", "critical_tickets_total"],
    "Satisfaction":      ["satisfaction_l3", "satisfaction_trend_3m", "satisfaction_min"],
    "Billing":           ["avg_monthly_spend", "late_payment_rate", "discount_rate", "spend_trend_3m"],
    "Value for money":   ["revenue_per_usage_hour", "price_per_tier", "data_utilisation"],
}
for g, cols in groups.items():
    print(f"  {g:<19} {', '.join(cols)}")

### Why these features

Four of them carry most of the signal, and each has a business reading:

| Feature | What it means to the business |
|---|---|
| `usage_trend_3m` | Last quarter's usage against the quarter before. Disengagement, measured. |
| `revenue_per_usage_hour` | Price paid per hour actually used. The **value gap** — the strongest single reason a customer decides they are overpaying. |
| `tickets_l6` | Chronic support pressure, not a one-off complaint. |
| `satisfaction_trend_3m` | Direction of travel, which matters more than the level. |

In [ ]:
checks = {
    "usage_trend_3m":         "Usage change vs prior quarter",
    "revenue_per_usage_hour": "$ paid per usage hour",
    "tickets_l6":             "Tickets per month (6m)",
    "satisfaction_l3":        "Recent satisfaction",
}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (col, label) in zip(axes, checks.items()):
    q = pd.qcut(train_tbl[col], 5, duplicates="drop")
    s = train_tbl.groupby(q, observed=True).churned.mean()
    ax.bar(range(len(s)), s.values * 100, color=SLATE)
    ax.set_title(label, fontsize=10)
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(["low", "", "mid", "", "high"][:len(s)])
    ax.set_ylabel("churn rate %")
plt.tight_layout()

Every one is monotone in the right direction. That is the sanity check to run *before*
training anything: if a feature has no gradient against the target here, it will not
suddenly acquire one inside a gradient booster.

In [ ]:
train_tbl.to_parquet("../data/processed/features_train.parquet", index=False)
score_tbl.to_parquet("../data/processed/features_score.parquet", index=False)
print("Saved features_train.parquet and features_score.parquet")

**Next:** `03_EDA.ipynb` — what the data says about why customers leave.